In [1]:
import yaml
import json

In [2]:
class helperfunction():
    def load_file(self,filepath):
        with open(filepath,'r',encoding="utf-8") as f:
            return f.read()
    def load_yaml(self,filepath):
        with open(filepath,'r',encoding="utf-8") as f:
            return yaml.safe_load(f)
    def fop(self,float_num):
        return float(f"{float_num:.1f}")
    def jsonstr(self,ip):
        return str(json.dumps(ip,indent=4, ensure_ascii=False))

In [3]:
hp = helperfunction()
resume_json = hp.load_file("resume_json.txt")
print(resume_json)

{
  "contact_information": {
    "name": "Surya Teja Menta",
    "email": "-",
    "phone": "+91 8309584461",
    "linkedin": "-",
    "jobdb_link": "-",
    "portfolio_link": "suryatejamenta.co.in"
  },
  "professional_summary": {
    "has_summary": "Yes",
    "summary_points": [
      "I’m Surya Teja Menta, Results-driven Senior Data Scientist with 4+ years of experience in Data Science, Machine Learning (ML), and Generative AI (GenAI).",
      "Proven expertise in RAG (Retrieval-Augmented Generation), LLM fine-tuning, MLOps, and end-to-end AI solutions.",
      "Strong background in data analytics, statistical modeling, AI-powered automation, and scalable AI architectures.",
      "IBM Certified Professional Data Scientist with hands-on experience in LangChain, Hugging Face, OpenAI APIs, Vector Databases (ChromaDB, Pinecone), and cloud deployments (AWS, GCP).",
      "Passionate about AI research, model optimization, and developing cutting-edge AI solutions."
    ]
  },
  "education

In [4]:
from google import genai

In [11]:
import json

class PromptBuilder(helperfunction):
    CRITERIA_FEWSHOT = {
        "Completeness": {
            5: (
                "Section contains all key elements from expected_content for this section with enough detail "
                "to understand the candidate's background and context. No major information gaps."
            ),
            3: (
                "Section contains some of the key elements but is missing 1–2 important parts or provides them "
                "only in shallow detail. Overall usable but not fully complete."
            ),
            1: (
                "Section is very sparse or missing most key elements; important information is absent or only hinted "
                "at. Hard to understand the candidate from this section alone."
            ),
        },

        "ContentQuality": {
            5: (
                "Follows clear Action → Method → Impact with quantifiable results; very specific; includes tools, "
                "methods, or techniques; shows strong, measurable improvement. Examples: social media with +35% "
                "engagement; Random Forest churn model (72%→88%, churn -20%); stakeholder management reducing "
                "escalations by 60%."
            ),
            3: (
                "Partially follows Action → Method → Impact; has some specifics but lacks clear, quantified results; "
                "reasonable but not strong. Examples: using Meta Business Suite to plan/publish content; churn model "
                "mentioned but no metrics; weekly stakeholder meetings to keep schedule."
            ),
            1: (
                "Very generic; only action with no method or impact; no tools, no metrics, low clarity. Examples: "
                "'Managed social media', 'Built a prediction model', 'Worked with stakeholders'."
            ),
        },

        "Grammar": {
            5: (
                "Grammar, spelling, and sentence structure are correct and natural; bullets are easy to read; verb tenses "
                "are consistent; only very minor or rare typos, if any."
            ),
            3: (
                "Some grammar or spelling issues, but the text remains understandable; occasional awkward phrasing or "
                "tense inconsistency, yet overall readability is acceptable."
            ),
            1: (
                'Frequent grammar and spelling mistakes; sentences are confusing or broken; tense usage is inconsistent; '
                "reader must work hard to interpret the meaning."
            ),
        },

        "Length": {
            5: (
                "Length is appropriate for the section: not too short, not overly long; information is dense and relevant; "
                "each sentence or bullet adds value without obvious redundancy."
            ),
            3: (
                "Length is somewhat suboptimal: either a bit short (missing some detail) or somewhat long with mild "
                "repetition or low-value bullets, but still usable."
            ),
            1: (
                "Length is clearly inappropriate: either extremely short (1–2 vague lines) or very long and repetitive "
                "with many low-information bullets or sentences."
            ),
        },

        "RoleRelevance": {
            5: (
                "Content is strongly aligned with the target role: tasks, tools, domain, and responsibilities clearly "
                "match what is expected for the <targetrole> role; shows appropriate seniority and impact."
            ),
            3: (
                "Content has partial alignment with the target role: some relevant skills or responsibilities, but mixed "
                "with less relevant tasks or not yet at the depth/seniority typically expected for <targetrole>."
            ),
            1: (
                "Content is mostly unrelated to the target role: focuses on other domains or generic duties; very little "
                "evidence that directly supports readiness for <targetrole>."
            ),
        },
    }


    def __init__(self, section, criteria, targetrole, cvresume):
        self.section    = section
        self.criteria   = criteria[::-1]  # keep your behavior
        self.cvresume   = cvresume
        self.targetrole = targetrole
        self.config     = self.load_yaml("prompt.yaml")

    def build_response_template(self):
        return {
            "section": self.section,
            "scores": {
                c: {"score": 0, "feedback": ""} for c in self.criteria
            }
        }

    def _build_criteria_block(self) -> str:
        """
        Build the Criteria section text, including few-shot examples
        for each criterion (score 5 / 3 / 1).
        """
        blocks = []
        for crit in self.criteria:
            fewshot = self.CRITERIA_FEWSHOT.get(crit)

            # Base line: just the criterion name
            block = f"- {crit}\n"

            # If we have few-shot examples, append them indented
            if fewshot:
                # order: 5, 3, 1
                for score in [5, 3, 1]:
                    if score in fewshot:
                        block += f"    score {score}: {fewshot[score]}\n"

            blocks.append(block)

        return "".join(blocks)

    def build(self):
        config_role      = self.config['role']['role1']
        config_objective = self.config['objective']['objective1']
        config_section   = self.config['section']['section1']
        config_expected  = self.config['expected_content'][self.section]
        config_scale     = self.config['scale']['score1']

        # 🔹 use helper to build criteria + few-shot
        criteria_block = self._build_criteria_block()

        prompt_role      = f"Role :\n{config_role}\n\n"
        prompt_objective = f"objectvie :\n{config_objective}\n"
        prompt_section   = f"section :\n{config_section}\n\n"
        prompt_expected  = f"expected :\n{config_expected}\n"
        prompt_criteria  = f"Criteria :\n{criteria_block}\n"
        prompt_scale     = f"Scale :\n{config_scale}\n"
        prompt_output    = f"output :\n{json.dumps(self.build_response_template(), indent=2)}\n\n"
        prompt_cvresume  = f"CV/Resume: \n{self.cvresume}\n"

        prompt = (
            prompt_role + prompt_objective + prompt_section
            + prompt_expected + prompt_criteria + prompt_scale
            + prompt_output + prompt_cvresume
        )

        prompt = prompt.replace("<section_name>", self.section)
        prompt = prompt.replace("<targetrole>", self.targetrole)
        return prompt


In [14]:
p1 = PromptBuilder( 
    section    = "Profile", 
    criteria   = ["Completeness", "ContentQuality"],
    targetrole = "Data science",
    cvresume   = resume_json
)
prompt1 = p1.build()
print(prompt1)

Role :
You are the expert HR evaluator

objectvie :
Evaluate the Profile section from the resume using the scoring criteria
Measure how well the candidate matches the Data science role.
Consider: degree relevance, experience alignment, skills/tools, and seniority evidence.
Score 0-5 using the scale above.

section :
You are evaluating the Profile section.

expected :
- Candidate's basic professional identity
- Clear positioning (e.g., "Data Analyst", "ML Engineer")
- Career direction or headline
- Avoid unnecessary personal details
- Feedback word 20 words

Criteria :
- ContentQuality
    score 5: Follows clear Action → Method → Impact with quantifiable results; very specific; includes tools, methods, or techniques; shows strong, measurable improvement. Examples: social media with +35% engagement; Random Forest churn model (72%→88%, churn -20%); stakeholder management reducing escalations by 60%.
    score 3: Partially follows Action → Method → Impact; has some specifics but lacks clea

In [15]:
p2 = PromptBuilder( 
    section  = "Summary", 
    criteria = ["Completeness", "ContentQuality","Grammar","Length","RoleRelevance"],
    targetrole = "Data science",
    cvresume = resume_json
)
prompt2 = p2.build()
print(prompt2)

Role :
You are the expert HR evaluator

objectvie :
Evaluate the Summary section from the resume using the scoring criteria
Measure how well the candidate matches the Data science role.
Consider: degree relevance, experience alignment, skills/tools, and seniority evidence.
Score 0-5 using the scale above.

section :
You are evaluating the Summary section.

expected :
- 2-4 sentence summary of experience
- Technical & domain strengths
- Career focus & value proposition
- Avoid buzzwords
- Feedback word 20 words

Criteria :
- RoleRelevance
    score 5: Content is strongly aligned with the target role: tasks, tools, domain, and responsibilities clearly match what is expected for the Data science role; shows appropriate seniority and impact.
    score 3: Content has partial alignment with the target role: some relevant skills or responsibilities, but mixed with less relevant tasks or not yet at the depth/seniority typically expected for Data science.
    score 1: Content is mostly unrelate

In [16]:
p3 = PromptBuilder( 
    section  = "Education", 
    criteria = ["Completeness","RoleRelevance"],
    targetrole = "Data science",
    cvresume = resume_json
)
prompt3 = p3.build()
print(prompt3)

Role :
You are the expert HR evaluator

objectvie :
Evaluate the Education section from the resume using the scoring criteria
Measure how well the candidate matches the Data science role.
Consider: degree relevance, experience alignment, skills/tools, and seniority evidence.
Score 0-5 using the scale above.

section :
You are evaluating the Education section.

expected :
- Institution name
- Degree & field of study
- Dates attended
- GPA, honors (optional)
- Relevance to data career
- Feedback word 20 words

Criteria :
- RoleRelevance
    score 5: Content is strongly aligned with the target role: tasks, tools, domain, and responsibilities clearly match what is expected for the Data science role; shows appropriate seniority and impact.
    score 3: Content has partial alignment with the target role: some relevant skills or responsibilities, but mixed with less relevant tasks or not yet at the depth/seniority typically expected for Data science.
    score 1: Content is mostly unrelated t

In [21]:
p4 = PromptBuilder( 
    section  = "Experience", 
    criteria = ["Completeness", "ContentQuality","Grammar","Length","RoleRelevance"],
    targetrole = "Data science",
    cvresume = "resume_json"
)
prompt4 = p4.build()
print(prompt4)

Role :
You are the expert HR evaluator

objectvie :
Evaluate the Experience section from the resume using the scoring criteria
Measure how well the candidate matches the Data science role.
Consider: degree relevance, experience alignment, skills/tools, and seniority evidence.
Score 0-5 using the scale above.

section :
You are evaluating the Experience section.

expected :
- Job title, employer, dates
- Clear bullet points
- Action → method → impact structure
- Technical tools used
- Quantifiable metrics
- Feedback word 20 words

Criteria :
- RoleRelevance
    score 5: Content is strongly aligned with the target role: tasks, tools, domain, and responsibilities clearly match what is expected for the Data science role; shows appropriate seniority and impact.
    score 3: Content has partial alignment with the target role: some relevant skills or responsibilities, but mixed with less relevant tasks or not yet at the depth/seniority typically expected for Data science.
    score 1: Content

In [24]:
p5 = PromptBuilder( 
    section  = "Activities", 
    criteria = ["Completeness", "ContentQuality","Grammar","Length"],
    targetrole = "Data science",
    cvresume = "resume_json"
)
prompt5 = p5.build()
print(prompt5)

Role :
You are the expert HR evaluator

objectvie :
Evaluate the Activities section from the resume using the scoring criteria
Measure how well the candidate matches the Data science role.
Consider: degree relevance, experience alignment, skills/tools, and seniority evidence.
Score 0-5 using the scale above.

section :
You are evaluating the Activities section.

expected :
- Competitions, hackathons, club activities
- Project descriptions with responsibilities
- Mention of tools/tech if applicable
- Feedback word 20 words

Criteria :
- Length
    score 5: Length is appropriate for the section: not too short, not overly long; information is dense and relevant; each sentence or bullet adds value without obvious redundancy.
    score 3: Length is somewhat suboptimal: either a bit short (missing some detail) or somewhat long with mild repetition or low-value bullets, but still usable.
    score 1: Length is clearly inappropriate: either extremely short (1–2 vague lines) or very long and re

In [26]:
p6 = PromptBuilder( 
    section  = "Skills", 
    criteria = ["Completeness","Length","RoleRelevance"],
    targetrole = "Data science",
    cvresume = "resume_json"
)
prompt6 = p6.build()
print(prompt6)

Role :
You are the expert HR evaluator

objectvie :
Evaluate the Skills section from the resume using the scoring criteria
Measure how well the candidate matches the Data science role.
Consider: degree relevance, experience alignment, skills/tools, and seniority evidence.
Score 0-5 using the scale above.

section :
You are evaluating the Skills section.

expected :
- Technical skills (Python, SQL, ML, Cloud)
- Tools (Power BI, Git, TensorFlow)
- Soft skills
- Language proficiency
- Clear grouping/categorization
- Feedback word 20 words

Criteria :
- RoleRelevance
    score 5: Content is strongly aligned with the target role: tasks, tools, domain, and responsibilities clearly match what is expected for the Data science role; shows appropriate seniority and impact.
    score 3: Content has partial alignment with the target role: some relevant skills or responsibilities, but mixed with less relevant tasks or not yet at the depth/seniority typically expected for Data science.
    score 1: 